# 08a_latency — 라우팅 지연 (EV-10)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 2.5절. 07b lock에서 고른 라우터마다 **질문 원문을 받아 시작 단계를 정할 때까지의 시간**을 같은 세션·같은 질문 200개(원천마다 50)로 잰다. 한 문항씩, 워밍업 20문항 뒤 중앙값과 p95를 낸다.

- 시간은 인코더(BGE 또는 각 라우터의 인코더) · 라우터 본체 · 보정과 결정 규칙으로 나눠 기록한다.
- BGE 라우터는 두 값을 남긴다: BGE까지 포함한 전체, 그리고 서비스가 캐시 때문에 BGE를 이미 계산했을 때 더해지는 지연.
- 지연은 구조와 데이터 크기로 정해지므로, 가중치를 남기지 않은 라우터(04 본체, 05 인코더)는 같은 클래스·설정·데이터 크기로 다시 만들어 잰다. ModernBERT와 RouteLLM 공개본은 실제 가중치를 쓴다.
- 보고 지표다. 효용 차이가 ±.01 안일 때만 짧은 쪽을 우선한다. Colab 값이라 실제 FastAPI 지연이 아니라 라우터끼리의 상대 비교로 쓴다.

**런타임: 2026.07(Python 3.12), T4 GPU.** CPU 측정(주)과 GPU 측정(참고)을 차례로 한다. 약 40분(추정). 07b lock 뒤에 실행한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'router-final-source-latency.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'ml/notebooks/router_final/requirements-common-colab.txt'),
                'sentencepiece==0.2.0'], check=True)

설치가 끝나면 **런타임 → 세션 다시 시작**을 한 번 한 뒤 아래 셀부터 실행합니다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
FINAL = RUN / 'latency'
assert (RUN / 'lock' / 'selection-lock.json').exists(), '07b lock이 먼저 있어야 한다'
import torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요'

def run_logged(script, args):
    """Run a script and stream its log into this cell."""
    proc = subprocess.Popen([sys.executable, str(REPO / 'ml/notebooks/router_final' / script), *map(str, args)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, PYTHONUNBUFFERED='1', TQDM_DISABLE='1'))
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

run_logged('common_track.py', ['fetch', '--external-dir', '/content/ev10-external'])
run_logged('finetune_track.py', ['fetch', '--external-dir', '/content/ev10-external-finetune'])

In [ ]:
# 1) CPU 측정(주 지표) → 2) GPU 측정(참고)
for device in ('cpu', 'cuda'):
    if (FINAL / f'latency-{device}.parquet').exists():
        print(device, '이미 있음'); continue
    run_logged('routing_latency.py', ['--run-dir', RUN, '--lock', RUN / 'lock' / 'selection-lock.json',
                              '--external-dir', '/content/ev10-external', '--external-finetune-dir', '/content/ev10-external-finetune',
                              '--output-dir', FINAL, '--device', device])

In [ ]:
# 3) 요약 (런타임을 다시 시작한 뒤에도 이 셀만 실행하면 된다)
from pathlib import Path
import pandas as pd
FINAL = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001') / 'latency'
pd.set_option('display.width', 200)
for device in ('cpu', 'cuda'):
    t = pd.read_parquet(FINAL / f'latency-{device}.parquet')
    t = t[t.stage.isin(['added_given_bge', 'total_with_bge', 'total', 'encoder'])]
    print(device); print(t.pivot_table(index='router', columns='stage', values=['median_ms', 'p95_ms']).round(1).to_string())